# Observatorio del Precio del Limón en Colima — Cuaderno de Limpieza y Preprocesamiento (E1)

Este cuaderno documenta e ilustra de manera interactiva el pipeline de limpieza, integración, preprocesamiento y control de calidad desarrollado para la entrega E1.

### Contenido:
1. **Integración de fuentes:** SNIIM diario y SIAP municipal.
2. **Valores faltantes:** Diagnóstico del mecanismo (MAR) e imputación del rezago 2026.
3. **Detección de atípicos y ruido:** Z-Score robusto (Hampel/MAD) e IQR.
4. **Normalización y estandarización:** Min-Max y Z-Score sobre registros poblacionales no sesgados.
5. **Discretización y jerarquías:** Frecuencias iguales vs. anchura igual y jerarquías temporales y espaciales.
6. **Reducción de datos:** Agregación y selección de variables por correlación ($|r| > 0.95$).
7. **Puerta de calidad (Pandera):** Certificación y verificación de conservación de filas.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Rutas relativas a la capa Plata
RAIZ = Path.cwd().parent.parent if Path.cwd().name == "E1" else Path.cwd()
PLATA = RAIZ / "data" / "plata"
CUARENTENA = RAIZ / "data" / "cuarentena"

print(f"Directorio raíz: {RAIZ}")
print(f"Capa Plata: {PLATA.exists()}")

## 1. Carga de Productos de Capa Plata

In [ ]:
sniim_d = pd.read_parquet(PLATA / "sniim_diario_homologado.parquet")
siap_m = pd.read_parquet(PLATA / "siap_municipal_homologado.parquet")
integrado_crudo = pd.read_parquet(PLATA / "integrado_sin_validar.parquet")
integrado_final = pd.read_parquet(PLATA / "integrado_reducido_validado.parquet")

print(f"SNIIM diario homologado: {sniim_d.shape}")
print(f"SIAP municipal homologado: {siap_m.shape}")
print(f"Integrado sin validar: {integrado_crudo.shape}")
print(f"Integrado reducido final validado: {integrado_final.shape}")

## 2. Diagnóstico del Mecanismo de Valores Faltantes (MAR)

Analizamos la distribución de valores faltantes en las variables del SIAP. Se comprueba que el 100% de los faltantes se concentran en el año 2026, lo que confirma un mecanismo **MAR (Missing at Random) de origen estructural** por el calendario de publicación del SIAP.

In [ ]:
tmp = integrado_crudo.assign(falta_siap=integrado_crudo["produccion_ton"].isna())
resumen_anio = tmp.groupby("anio")["falta_siap"].agg(total="count", faltantes="sum", porcentaje=lambda x: f"{x.mean()*100:.1f}%")
print("Faltantes de SIAP por año en el conjunto integrado:")
print(resumen_anio)

## 3. Detección de Atípicos (Filtro Hampel / Z-Score Robusto e Incoherencias)

Se evalúa el reporte generado por el filtro Hampel sobre las cotizaciones diarias de SNIIM.

In [ ]:
reporte_atipicos = pd.read_csv(PLATA / "reporte_atipicos_sniim.csv")
print(f"Total registros en reporte de atípicos e incoherencias: {len(reporte_atipicos)}")
print(f"Atípicos estadísticos (Hampel): {reporte_atipicos['es_atipico'].sum()}")
print(f"Incoherencias lógicas (min > frec): {reporte_atipicos['precio_incoherente'].sum()}")

reporte_atipicos[reporte_atipicos['precio_incoherente']][['fecha', 'origen_estado', 'destino_mercado', 'precio_min', 'precio_frec', 'precio_max']]

## 4. Reducción de Variables por Matriz de Correlación

Evaluación de colinealidad estricta entre las variables agrícolas ($|r| > 0.95$).

In [ ]:
candidatas = ["produccion_ton", "sembrada_ha", "cosechada_ha", "valor_produccion"]
corr = siap_m[candidatas].corr()
print("Matriz de Correlación de Pearson (SIAP municipal):")
print(corr.round(3))
print("\nVariables con r > 0.95 respecto a produccion_ton: sembrada_ha y cosechada_ha (eliminadas)")

## 5. Puerta de Calidad y Registro de Cuarentena (Pandera)

Verificación de registros que pasaron la puerta y registro aislado en cuarentena.

In [ ]:
cuarentena = pd.read_csv(CUARENTENA / "sniim_diario.csv")
print(f"Registros en cuarentena: {len(cuarentena)}")
print(f"Motivo documentado: {cuarentena.loc[0, 'motivo']}")
print(f"ID SHA-256 idempotente: {cuarentena.loc[0, 'registro_id']}")

# Comprobación de la conservación de registros:
total_sniim = len(sniim_d)
validos_sniim = len(pd.read_parquet(PLATA / "sniim_diario_validado.parquet"))
cuarentena_sniim = len(cuarentena)
print(f"\nConservación de registros SNIIM: {total_sniim} entrada == {validos_sniim} válidos + {cuarentena_sniim} cuarentena -> {total_sniim == validos_sniim + cuarentena_sniim}")